In [11]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

In [12]:
# 1. 데이터 불러오기
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/car.data", header=None)
df.columns = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']
# 첫 번째 데이터 행이 컬럼 이름으로 사용되어 데이터 1개가 손실되는 것을 막기 위해 header=None 옵션 추
print(df.head())
print(df.shape)

  buying  maint doors persons lug_boot safety  class
0  vhigh  vhigh     2       2    small    low  unacc
1  vhigh  vhigh     2       2    small    med  unacc
2  vhigh  vhigh     2       2    small   high  unacc
3  vhigh  vhigh     2       2      med    low  unacc
4  vhigh  vhigh     2       2      med    med  unacc
(1728, 7)


In [13]:
label_encoders = {}

for column in df.columns:
    label_encoders[column] = LabelEncoder()
    df[column] = label_encoders[column].fit_transform(df[column])

In [14]:
# 2. 입력/정답 데이터 분리
X = df.drop("class", axis=1).values
y = df["class"].values

y=pd.get_dummies(y).values

In [15]:
# 3. 학습 / 테스트 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [16]:
# 4. 데이터 정규화
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [17]:
# 5. 모델 생성
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(64, activation="relu", input_shape=(X_train.shape[1],)))
model.add(tf.keras.layers.Dense(32, activation="relu"))
model.add(tf.keras.layers.Dense(16, activation="relu"))
model.add(tf.keras.layers.Dense(4, activation="softmax"))
# 성능을 높이기 위해 은닉층 하나 더 추가
# unacc,acc,good,v-good 중에 하나를 고르는 다중 분류문제이므로 (4, activation="softmax")으로 변경

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [18]:
# 6. 모델 설정
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
#다중 분류 문제이므로 손실함수 categorical_crossentropy 사용

In [19]:
# 7. 모델 구조 출력
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 64)             │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,124 (12.20 KB)

 Trainable params: 3,124 (12.20 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
# 8. 모델 학습
history = model.fit(X_train, y_train, epochs=100, batch_size=32, validation_split=0.2, verbose=1)

Epoch 1/100
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.6480 - loss: 1.0947 - val_accuracy: 0.7040 - val_loss: 0.8161
Epoch 2/100
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6995 - loss: 0.7549 - val_accuracy: 0.7040 - val_loss: 0.6818
Epoch 3/100
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7050 - loss: 0.6583 - val_accuracy: 0.7112 - val_loss: 0.6328
Epoch 4/100
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7267 - loss: 0.6005 - val_accuracy: 0.7329 - val_loss: 0.5741
Epoch 5/100
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7466 - loss: 0.5439 - val_accuracy: 0.7726 - val_loss: 0.5211
Epoch 6/100
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7919 - loss: 0.4807 - val_accuracy: 0.7942 - val_loss: 0.4532
Epoch 7/100
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8217 - loss: 0.4164 - val_accuracy: 0.8448 - val_loss: 0.3966
Epoch 8/100
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8534 - loss: 0.3644 - val_accuracy: 0.8520 - 

In [21]:
# 9. 테스트 데이터 예측
# softmax 출력값(4개 클래스별 확률)에서 가장 높은 확률의 클래스 인덱스(0, 1, 2, 3) 추출
y_prob = model.predict(X_test)
y_pred = np.argmax(y_prob, axis=1)

# 평가 지표 계산을 위해 원-핫 인코딩된 y_test도 1차원 라벨로 변환
y_test_labels = np.argmax(y_test, axis=1)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 


In [22]:
# 10. 성능 평가
# LabelEncoder로 인코딩된 클래스 원래 이름 가져오기
target_names = label_encoders['class'].classes_

accuracy = accuracy_score(y_test_labels, y_pred)
print("\nAccuracy =", accuracy)
print("\nConfusion Matrix")
print(confusion_matrix(y_test_labels, y_pred))
print("\nClassification Report")
print(classification_report(y_test_labels, y_pred, target_names=target_names))


Accuracy = 0.9942196531791907

Confusion Matrix
[[ 75   1   1   0]
 [  0  14   0   0]
 [  0   0 242   0]
 [  0   0   0  13]]

Classification Report
              precision    recall  f1-score   support

         acc       1.00      0.97      0.99        77
        good       0.93      1.00      0.97        14
       unacc       1.00      1.00      1.00       242
       vgood       1.00      1.00      1.00        13

    accuracy                           0.99       346
   macro avg       0.98      0.99      0.99       346
weighted avg       0.99      0.99      0.99       346

